In [1]:
import os

from dotenv import load_dotenv

# Explicitly providing path to '.env'
from pathlib import Path  # Python 3.6+ only
# Load .env variables
_ = load_dotenv(dotenv_path=f"{Path().resolve().parents[1]}/src/.env")

# with the new api
from importnb import imports
with imports("ipynb"):
    from utils import to_timestamp, df_tangara_sensors, df_to_csv

PM2.5: 35.9, AQI: 102
PM2.5: 35.9, Measure Level: MeasureLevels.UNHEALTHY_FOR_SENSITIVE_GROUPS, Range Values: Min: 35.5, Max: 55.4
AQI: 102, Measure Level: MeasureLevels.UNHEALTHY_FOR_SENSITIVE_GROUPS, Range Values: Min: 101, Max: 150


## Tangara Sensors

In [2]:
# Start Date Time ISO 8601 Format, TZ='America/Bogota' -05:00
START_ISO8601_DATETIME=os.getenv("START_ISO8601_DATETIME", None)
start_timestamp = to_timestamp(START_ISO8601_DATETIME)
# End Date Time ISO 8601 Format, TZ='America/Bogota' -05:00
END_ISO8601_DATETIME=os.getenv("END_ISO8601_DATETIME", None)
end_timestamp = to_timestamp(os.getenv("END_ISO8601_DATETIME", None))

print(f'Since: {START_ISO8601_DATETIME} -> {start_timestamp}, Until: {END_ISO8601_DATETIME} -> {end_timestamp}')

2024-03-01 17:19:05.143 | DEBUG    | utils:to_timestamp:99 - datetime_iso8601: 2024-03-01T00:00:00-05:00, Timestamp: 1709269200000
2024-03-01 17:19:05.144 | DEBUG    | utils:to_timestamp:99 - datetime_iso8601: 2024-03-01T23:59:59-05:00, Timestamp: 1709355599000


Since: 2024-03-01T00:00:00-05:00 -> 1709269200000, Until: 2024-03-01T23:59:59-05:00 -> 1709355599000


In [3]:
# Data Frame Tangaras from InfluxDB
df_tangaras = df_tangara_sensors(start_timestamp, end_timestamp)
df_tangaras.drop_duplicates(subset=['MAC'], inplace=True)

print(f"Period of Time: Since: {START_ISO8601_DATETIME}, Until: {END_ISO8601_DATETIME}")
print(f"Total Tangara Sensors: {len(df_tangaras)}")

df_tangaras.head()

2024-03-01 17:19:05.149 | DEBUG    | utils:query_tangaras:156 - sql_query: SELECT DISTINCT(geo) AS "geohash" FROM "fixed_stations_01" WHERE ("geo3" = 'd29') AND time >= 1709269200000ms AND time <= 1709355599000ms GROUP BY "name";
2024-03-01 17:19:05.274 | DEBUG    | utils:request_influxdb:131 - response: <Response [200]>
2024-03-01 17:19:05.314 | DEBUG    | utils:df_tangara_sensors:406 - Data Frame Tangaras Sensors: <class 'pandas.core.frame.DataFrame'>
Index: 16 entries, TANGARA_2BBA to TANGARA_06BE
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   GEOHASH      16 non-null     object
 1   MAC          16 non-null     object
 2   GEOLOCATION  16 non-null     object
 3   LATITUDE     16 non-null     object
 4   LONGITUDE    16 non-null     object
dtypes: object(5)
memory usage: 768.0+ bytes



Period of Time: Since: 2024-03-01T00:00:00-05:00, Until: 2024-03-01T23:59:59-05:00
Total Tangara Sensors: 16


,GEOHASH,MAC,GEOLOCATION,LATITUDE,LONGITUDE
ID,,,,,
TANGARA_2BBA,d29e6b4,D29ESP32DE02BBA,3.3844757080078125 -76.51634216308594,3.3844757080078125,-76.51634216308594
TANGARA_2BDE,d29e6de,D29ESP32DE52BDE,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_3B7E,d29ee40,D29ESP32DE53B7E,3.4394073486328125 -76.50810241699219,3.4394073486328125,-76.50810241699219
TANGARA_421A,d29e6de,D29ESP32DE5421A,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_422A,d29e6de,D29ESP32DE5422A,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531


In [4]:
# Save Tangaras into CSV file
df_to_csv(df_tangaras, "tangaras.csv")

2024-03-01 17:19:05.329 | DEBUG    | utils:df_to_csv:311 - Save DataFrame: /home/sebaxtian/Workspaces/Tangara/tangara-evaluation/src/data/0_raw/tangaras.csv
